# Lab | Embeddings
## Find the answer before the customer finishes typing

You work for a digital bank. Every day, thousands of customers write to support: *my card has not arrived*, *I was charged twice*, *why is the exchange rate different?* Most of these questions have been asked, and answered, many times before.

The support team wants a tool that takes a new message and instantly shows the agent the **most similar past messages** and their category, so the agent can reuse the answer. Later, the same search will feed a chatbot (that is Day 5's RAG). You will build that tool with **sentence embeddings** and check, with numbers, whether it beats a classic TF-IDF search.

**The data** is the Banking77 dataset: 13,083 real customer messages to a bank, each labelled with one of 77 categories (called **intents**), such as `card_arrival` or `exchange_rate`. `data/banking_train.csv` has 10,003 messages, which play the role of the past messages, and `data/banking_test.csv` has 3,080 messages, which play the role of new ones.

**This lab practises** semantic search with sentence embeddings. You will use, from today's **Embeddings and Semantic Search** lesson: `SentenceTransformer.encode`, cosine similarity, unit-length vectors and the dot product, top-*k* search with `np.argsort`, the TF-IDF search, and KMeans with the adjusted Rand index.

**By the end, you can:**

1. Implement a top-*k* semantic search with sentence embeddings and cosine similarity.
2. Evaluate it against a TF-IDF baseline and explain where each method fails.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on.

## Setup

The first run downloads the `all-MiniLM-L6-v2` model (about 90 MB, internet needed once). If the import fails, install the library from a terminal with `pip install sentence-transformers`.

In [1]:
import os
import logging
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"           # on my Mac KMeans crashes (OMP: Error #179) when torch is loaded, this fixes it
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

pd.set_option("display.max_colwidth", 90)

In [2]:
train = pd.read_csv("../data/banking_train.csv")
test = pd.read_csv("../data/banking_test.csv")
print(train.shape, test.shape, train["category"].nunique())
train.head()

(10003, 2) (3080, 2) 77


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived after 2 weeks?,card_arrival
2,I have been waiting over a week. Is the card still coming?,card_arrival
3,Can I track my card while it is in the process of delivery?,card_arrival
4,"How do I know if I will get my card, or if it is lost?",card_arrival


**Checkpoint:** `(10003, 2) (3080, 2) 77`, and two columns, `text` and `category`.

## Explore the data

1. How many training messages does each intent have? Look at the smallest and the largest with `value_counts()`.
2. Print 5 random training messages with their intent (`sample(5, random_state=0)`).
3. Print all the intents whose name contains `card` (`train["category"].unique()` and a list comprehension). How easy do you think it is to tell them apart?

In [3]:
# 1. Messages per intent
counts = train["category"].value_counts()
print(counts.describe().round(1))
display(pd.concat([counts.head(5), counts.tail(5)]))

count     77.0
mean     129.9
std       32.9
min       35.0
25%      112.0
50%      127.0
75%      159.0
max      187.0
Name: count, dtype: float64


category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, dtype: int64

In [4]:
# 2. Five random training messages
train.sample(5, random_state=0)

,text,category
7145,How can I transfer money to a beneficiary?,beneficiary_not_allowed
834,How is my ATM deposit not available yet?,pending_cash_withdrawal
3585,There is a vendor name I don't recognize on a payment from my account. I don't think I...,card_payment_not_recognised
4497,My statement shows charges for things I never purchased. Were my account details stolen?,compromised_card
7046,"Hi, I am interested buying crypto currency but unable to purchase it through the appli...",beneficiary_not_allowed


In [5]:
# 3. Intents with "card" in their name
card_intents = [c for c in train["category"].unique() if "card" in c]
print(len(card_intents), "of", train["category"].nunique(), "intents")
sorted(card_intents)

27 of 77 intents


['activate_my_card',
 'card_about_to_expire',
 'card_acceptance',
 'card_arrival',
 'card_delivery_estimate',
 'card_linking',
 'card_not_working',
 'card_payment_fee_charged',
 'card_payment_not_recognised',
 'card_payment_wrong_exchange_rate',
 'card_swallowed',
 'compromised_card',
 'declined_card_payment',
 'disposable_card_limits',
 'get_disposable_virtual_card',
 'get_physical_card',
 'getting_spare_card',
 'getting_virtual_card',
 'lost_or_stolen_card',
 'order_physical_card',
 'pending_card_payment',
 'reverted_card_payment?',
 'supported_cards_and_currencies',
 'top_up_by_card_charge',
 'topping_up_by_card',
 'virtual_card_not_working',
 'visa_or_mastercard']

**Question:** what did you notice about the intents with `card` in their name?

27 of the 77 intents have `card` in their name, so more than a third of the data is about cards. Many of them are very close to each other, and I'm not sure I could always tell them apart myself:

- `card_arrival` vs `card_delivery_estimate` (*where is my card?* vs *when will my card come?*)
- `get_physical_card` vs `order_physical_card`, `getting_virtual_card` vs `get_disposable_virtual_card`
- `lost_or_stolen_card` vs `compromised_card`
- `card_not_working` vs `virtual_card_not_working` vs `declined_card_payment`
- `card_payment_fee_charged` vs `top_up_by_card_charge`

Also one intent is called `reverted_card_payment?`, with a question mark in its name.

The intents are not balanced: from 35 messages (`contactless_not_working`) to 187 (`card_payment_fee_charged`), median 127. And the random sample already shows some questionable labels: *How is my ATM deposit not available yet?* is labelled `pending_cash_withdrawal` (a deposit, not a withdrawal), and the crypto question is `beneficiary_not_allowed`. So I expect the search to make mistakes that are partly the labels' fault.

## Challenge 1: compare sentences

Start small. Load the model and measure how similar a few pairs of customer messages are.

1. Load `all-MiniLM-L6-v2` into a variable called `model`.
2. For every pair in `pairs`, encode the two sentences and print their cosine similarity, rounded to 2 decimals, next to the pair.

In [6]:
pairs = [
    ("My card still hasn't arrived", "When will I receive my new card?"),
    ("I was charged twice for the same purchase", "Why is there a duplicate payment on my statement?"),
    ("I was charged twice for the same purchase", "How do I change my PIN?"),
    ("How do I top up my account?", "My top up failed"),
    ("My card was stolen", "My card was not stolen"),
]

# 1. Load the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Similarity of each pair
for a, b in pairs:
    vectors = model.encode([a, b])
    print(f"{cosine_similarity(vectors)[0, 1]:.2f}  {a!r} / {b!r}")

Could not fetch README.md from 'sentence-transformers/all-MiniLM-L6-v2' (TypeError: Decompressor.decompress() got an unexpected keyword argument 'output_buffer_limit'). Model will load without a model card.


0.65  "My card still hasn't arrived" / 'When will I receive my new card?'
0.60  'I was charged twice for the same purchase' / 'Why is there a duplicate payment on my statement?'
0.13  'I was charged twice for the same purchase' / 'How do I change my PIN?'
0.63  'How do I top up my account?' / 'My top up failed'
0.96  'My card was stolen' / 'My card was not stolen'


**Checkpoint:** the duplicate-payment pair scores about 0.6 although it shares almost no words, the PIN pair scores about 0.1, and the last pair scores above 0.9.

**Question:** the last pair scores higher than any other. Is that a problem for a support tool? What would happen if a bot answered *My card was not stolen* with the answer to *My card was stolen*?

Yes, that's a real problem. *Stolen* and *not stolen* score 0.96, higher than the duplicate-payment pair (0.60), which actually means the same thing. The model sees that both sentences are about a stolen card and mostly ignores the *not*. So similarity here means "about the same topic", not "says the same thing".

If a bot answered *My card was not stolen* with the answer for a stolen card, it would tell the customer to block the card, or block it right away. In the best case the customer is confused, in the worst case their working card is blocked while they are abroad. For a search that shows past messages to a **human agent** it's less bad, since the agent reads the message and sees the *not*. For an automatic answer it's dangerous, at least for intents with consequences like blocking cards or reporting fraud. There the bot should ask back or pass the message to a person.

## Challenge 2: build the index

A search engine encodes all the documents once and keeps the vectors.

1. Encode all the training messages (`train["text"].tolist()`) into `train_vectors`. Use `normalize_embeddings=True` so that every vector has length 1, and `batch_size=64`. It takes between 10 seconds and a minute, depending on your computer.
2. Print the shape of `train_vectors` and the length of the first vector (`np.linalg.norm`).

In [7]:
# 1. Encode the training messages
train_vectors = model.encode(train["text"].tolist(), normalize_embeddings=True, batch_size=64)

# 2. Shape and length
print(train_vectors.shape, np.linalg.norm(train_vectors[0]).round(4))

(10003, 384) 1.0


**Checkpoint:** `(10003, 384) 1.0`.

## Challenge 3: semantic search

Complete `semantic_search(query, k=5)`. It must return a DataFrame with the `k` most similar training messages, best first, and three columns: `score`, `category` and `text`.

The steps, as in the lesson:

1. Encode the query with `normalize_embeddings=True`.
2. Compute the similarity with every training message. Because all vectors have length 1, the dot product **is** the cosine similarity: `train_vectors @ query_vector`.
3. Find the positions of the `k` highest scores with `np.argsort` (it sorts from lowest to highest).
4. Build the DataFrame from those positions.

In [8]:
def semantic_search(query, k=5):
    # 1. Encode the query
    query_vector = model.encode(query, normalize_embeddings=True)

    # 2. Similarity with every training message
    scores = train_vectors @ query_vector

    # 3. Positions of the k highest scores
    top = np.argsort(scores)[::-1][:k]

    # 4. DataFrame with score, category and text
    return pd.DataFrame({
        "score": scores[top].round(3),
        "category": train["category"].iloc[top].values,
        "text": train["text"].iloc[top].values,
    })


semantic_search("I can't find my card anywhere")

,score,category,text
0,0.953,lost_or_stolen_card,I cant find my card.
1,0.925,lost_or_stolen_card,I can't find my card! Can you help?
2,0.922,lost_or_stolen_card,"I can't find my card, it is lost."
3,0.915,lost_or_stolen_card,I can't find my card! What should I do?
4,0.903,card_arrival,Where is my card?


**Checkpoint:** the first four results are lost-card messages with the intent `lost_or_stolen_card` and scores above 0.9. The fifth, *Where is my card?*, belongs to `card_arrival`.

## Challenge 4: a TF-IDF baseline

Is the embedding model worth it? Build the Day 1 alternative and compare.

1. Fit a `TfidfVectorizer()` (default settings) on the training texts and keep the training matrix in `train_tfidf`.
2. Write `tfidf_search(query, k=5)`, returning the same three columns as `semantic_search`. TF-IDF rows already have length 1, so the dot product is again the cosine similarity. The product of two sparse matrices is sparse: `.toarray().ravel()` turns the result into a flat NumPy array.
3. Run both searches on each query in `queries`.

In [9]:
queries = [
    "I can't find my card anywhere",
    "Someone took money from my account without asking",
    "Why did the exchange rate change so much?",
    "The ATM gave me less cash than I asked for",
]

# 1. Fit the vectoriser
tfidf = TfidfVectorizer()
train_tfidf = tfidf.fit_transform(train["text"])

# 2. tfidf_search
def tfidf_search(query, k=5):
    scores = (train_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        "score": scores[top].round(3),
        "category": train["category"].iloc[top].values,
        "text": train["text"].iloc[top].values,
    })

# 3. Compare
for query in queries:
    print(f"Query: {query!r}\nSemantic search")
    display(semantic_search(query))
    print("TF-IDF search")
    display(tfidf_search(query))

Query: "I can't find my card anywhere"
Semantic search


,score,category,text
0,0.953,lost_or_stolen_card,I cant find my card.
1,0.925,lost_or_stolen_card,I can't find my card! Can you help?
2,0.922,lost_or_stolen_card,"I can't find my card, it is lost."
3,0.915,lost_or_stolen_card,I can't find my card! What should I do?
4,0.903,card_arrival,Where is my card?


TF-IDF search


,score,category,text
0,0.873,get_physical_card,I can't find my PIN anywhere.
1,0.739,card_acceptance,Can I use my card anywhere?
2,0.695,order_physical_card,Can I receive my card anywhere?
3,0.668,card_acceptance,Is there anywhere I can't use my card?
4,0.653,card_acceptance,Can I pay with my card anywhere?


Query: 'Someone took money from my account without asking'
Semantic search


,score,category,text
0,0.814,card_payment_not_recognised,Someone else accessed my account and made a payment.
1,0.782,cash_withdrawal_not_recognised,"Someone withdrew money from my account, and it wasn't me."
2,0.754,card_payment_not_recognised,A payment in my account was made by someone I don't know
3,0.739,cash_withdrawal_not_recognised,"There was money taken out of my account, and I'm pretty sure I didn't do it."
4,0.713,cash_withdrawal_not_recognised,there is money in my account that i didnt put there


... (6 outputs left out)

Query: 'The ATM gave me less cash than I asked for'
Semantic search


,score,category,text
0,0.978,wrong_amount_of_cash_received,The ATM gave me less cash than requested
1,0.929,wrong_amount_of_cash_received,I got less cash at the ATM than I asked for. Why?
2,0.914,wrong_amount_of_cash_received,Why did I get less cash than what I asked in the ATM?
3,0.908,wrong_amount_of_cash_received,The ATM did not give me as much cash as I requested.
4,0.907,wrong_amount_of_cash_received,Why did the ATM give me less money than what I requested?


TF-IDF search


,score,category,text
0,0.789,wrong_amount_of_cash_received,The ATM gave me less cash than requested
1,0.753,wrong_amount_of_cash_received,I got less cash at the ATM than I asked for. Why?
2,0.718,wrong_amount_of_cash_received,Why did I get less cash than what I asked in the ATM?
3,0.717,wrong_amount_of_cash_received,"I got less than what i asked for in the ATM, why?"
4,0.712,wrong_amount_of_cash_received,How can I fix this? The ATM gave me more cash than I'd asked for.


**Checkpoint:** for the first query, TF-IDF's top result is *I can't find my PIN anywhere*.

**Question:** for which queries does TF-IDF go wrong, and why? For which does it do as well as semantic search? Write one query of your own where you expect TF-IDF to fail, and check it.

**TF-IDF goes wrong on the first two queries:**

- *I can't find my card anywhere*: TF-IDF puts *I can't find my **PIN** anywhere* on rank 1 and fills the rest with `card_acceptance` (*Can I use my card **anywhere**?*). *Anywhere* is a rare word, so it gets a high weight, while *card* appears in thousands of messages and counts for almost nothing. The rarest shared word wins, whatever it means. Semantic search returns four `lost_or_stolen_card` messages above 0.9.
- *Someone took money from my account without asking*: TF-IDF finds *Someone **took** my card **without** my permission* (`lost_or_stolen_card`) and *I am **asking** to end my account* (`terminate_account`). It matches *took*, *without* and *asking* in the wrong meaning. Semantic search finds unrecognised payments and withdrawals, which is what the customer means, even though *took money without asking* appears in none of them.

**Where TF-IDF does as well:**

- *The ATM gave me less cash than I asked for*: both return five `wrong_amount_of_cash_received` messages. The query is almost word for word a training message (*The ATM gave me less cash than requested*), so counting words is enough.
- *Why did the exchange rate change so much?*: both put the same message on rank 1 (*Why did my exchange rate change?*). Both then mix three exchange-rate intents (`exchange_rate`, `card_payment_wrong_exchange_rate`, `wrong_exchange_rate_for_cash_withdrawal`), because the query does not say whether it is about a card payment or a cash withdrawal. That's ambiguous for any method.

My own queries: *A shop billed me two times* uses none of the usual words (*charged*, *twice*, *duplicate*), so I expect TF-IDF to fail. And as a counter-test, *The machine kept my plastic*, where I'm not sure the model knows that *plastic* means card:

In [10]:
for query in ["A shop billed me two times", "The machine kept my plastic"]:
    print(f"Query: {query!r}\nSemantic search")
    display(semantic_search(query, k=3))
    print("TF-IDF search")
    display(tfidf_search(query, k=3))

Query: 'A shop billed me two times'
Semantic search


,score,category,text
0,0.734,transaction_charged_twice,I was charged multiple times for one purchase
1,0.718,transaction_charged_twice,My account was charged twice for one purchase
2,0.709,transaction_charged_twice,I was charged twice for the same purchase!


TF-IDF search


,score,category,text
0,0.437,extra_charge_on_statement,I got billed an extra pound!
1,0.380,transaction_charged_twice,I got charged two times for a transaction
2,0.363,transaction_charged_twice,There is a payment that I was charged two times for.


Query: 'The machine kept my plastic'
Semantic search


,score,category,text
0,0.452,wrong_amount_of_cash_received,It kept my cash
1,0.446,wrong_amount_of_cash_received,The machine gave me the wrong amount of cash.
2,0.390,card_swallowed,The ATM machine ate my card.


TF-IDF search


,score,category,text
0,0.689,card_swallowed,The ATM kept my card?
1,0.659,wrong_amount_of_cash_received,It kept my cash
2,0.507,card_swallowed,what do I need to do if the ATM kept my card?


*A shop billed me two times*: as expected. Semantic search returns three `transaction_charged_twice` messages (~0.72), TF-IDF matches *billed* and returns *I got **billed** an extra pound!* (`extra_charge_on_statement`) on rank 1.

*The machine kept my plastic*: here it's the other way around. Semantic search puts two `wrong_amount_of_cash_received` messages first, with low scores (~0.45), and the right one (*The ATM machine ate my card*, `card_swallowed`) only on rank 3. Apparently it connects *machine* to an ATM, but not *plastic* to a card. TF-IDF finds *The ATM **kept** my card?* (`card_swallowed`) because *kept* is a rare word that happens to be the key word here. So TF-IDF is not useless, it just only works when the customer uses the same words as the past messages. The low semantic scores are a hint that the search is unsure (see the extra task).

## Challenge 5: measure it

Four queries are anecdotes. To compare the methods properly, treat the 3,080 test messages as new customer messages, find the most similar training message for each one, and check whether it has the same intent. The share of correct intents is the **top-1 accuracy** of the search.

1. Encode the test messages into `test_vectors` (normalised, as before).
2. `test_vectors @ train_vectors.T` gives a 3,080 × 10,003 matrix with the similarity of every test message to every training message. For each row, `.argmax(axis=1)` gives the position of the most similar training message. Use it to get the predicted intents and compute the accuracy.
3. Do the same with TF-IDF. The similarity matrix is sparse here, and its `argmax(axis=1)` returns a matrix: wrap it with `np.asarray(...).ravel()`.
4. Print both accuracies.

In [11]:
# 1. Encode the test messages
test_vectors = model.encode(test["text"].tolist(), normalize_embeddings=True, batch_size=64)

# 2. Embedding accuracy
similarity = test_vectors @ train_vectors.T
predicted = train["category"].to_numpy()[similarity.argmax(axis=1)]
embedding_accuracy = (predicted == test["category"].to_numpy()).mean()

# 3. TF-IDF accuracy
test_tfidf = tfidf.transform(test["text"])
best_tfidf = np.asarray((test_tfidf @ train_tfidf.T).argmax(axis=1)).ravel()
predicted_tfidf = train["category"].to_numpy()[best_tfidf]
tfidf_accuracy = (predicted_tfidf == test["category"].to_numpy()).mean()

# 4. Print both
print(f"Top-1 accuracy, embeddings: {embedding_accuracy:.3f}")
print(f"Top-1 accuracy, TF-IDF:     {tfidf_accuracy:.3f}")

Top-1 accuracy, embeddings: 0.920
Top-1 accuracy, TF-IDF:     0.791


**Checkpoint:** about 0.92 for embeddings and 0.79 for TF-IDF.

5. Now look at the mistakes of the embedding search. Build a DataFrame with the test text, the true intent and the predicted intent, keep the rows where they differ, and print the 8 most frequent (true, predicted) pairs with `value_counts()`. Then print a few of the mistaken messages.

In [12]:
# 5. Mistakes of the embedding search
results = pd.DataFrame({"text": test["text"], "true": test["category"], "predicted": predicted})
mistakes = results[results["true"] != results["predicted"]]
print(len(mistakes), "mistakes of", len(results))
display(mistakes.value_counts(["true", "predicted"]).head(8))

top_confusions = mistakes.value_counts(["true", "predicted"]).head(4).index
for true, pred in top_confusions:
    display(mistakes[(mistakes["true"] == true) & (mistakes["predicted"] == pred)].head(3))

245 mistakes of 3080


true                         predicted                              
top_up_failed                top_up_reverted                            6
pin_blocked                  change_pin                                 5
verify_my_identity           why_verify_identity                        4
exchange_via_app             fiat_currency_support                      4
pending_transfer             balance_not_updated_after_bank_transfer    4
declined_transfer            declined_card_payment                      4
card_payment_not_recognised  compromised_card                           4
card_delivery_estimate       card_arrival                               4
Name: count, dtype: int64

,text,true,predicted
2642,I have a top up that didn't go through. Why?,top_up_failed,top_up_reverted
2649,Why didn`t my topup go through?,top_up_failed,top_up_reverted
2654,Why caused my top up to not work?,top_up_failed,top_up_reverted


,text,true,predicted
522,Will you reinstate my PIN?,pin_blocked,change_pin
523,I accidentally blocked my PIN. How do I reset it?,pin_blocked,change_pin
527,"How do I reset my PIN, I can't seem to use my card?",pin_blocked,change_pin


,text,true,predicted
3000,What can I use to verify my identify?,verify_my_identity,why_verify_identity
3003,How can I verify my indentity?,verify_my_identity,why_verify_identity
3020,How does my identity get verified?,verify_my_identity,why_verify_identity


,text,true,predicted
402,Can I change from one currency to another?,exchange_via_app,fiat_currency_support
410,What all currencies can be exchanged?,exchange_via_app,fiat_currency_support
428,What currency's can i exchange for?,exchange_via_app,fiat_currency_support


**Question:** look at the most frequent confusions. Are they really mistakes? What does this tell you about the data and about the 0.92?

Mostly they are not real mistakes. The top confusions are pairs of intents that are almost the same:

- `top_up_failed` → `top_up_reverted` (6): *Why didn't my topup go through?* could honestly be both.
- `pin_blocked` → `change_pin` (5): *I accidentally blocked my PIN. How do I reset it?* is about blocking **and** about changing the PIN.
- `verify_my_identity` → `why_verify_identity` (4): *How does my identity get verified?* is more *why/how* than *I want to verify*.
- `exchange_via_app` → `fiat_currency_support` (4): *What currencies can be exchanged?* is literally a question about supported currencies.
- `card_delivery_estimate` → `card_arrival` (4), `card_payment_not_recognised` → `compromised_card` (4): the same pairs I already noticed when exploring the data.

No confusion pair has more than 6 messages: the 245 mistakes are spread thin over many pairs of neighbouring intents, not one big systematic error. For an agent the answer of the "wrong" neighbour would often still help.

What it tells me about the data: the intent boundaries are partly arbitrary and some labels are noisy (as seen in the sample earlier). So no method can get 100% here, and the 0.92 is closer to the ceiling than it looks. On the other side, many test messages are close rewordings of training messages from the same templates. Real customers write more differently (typos, slang, several problems in one message, like *the machine kept my plastic*), so I would expect the real accuracy to be lower than 0.92.

## Challenge 6: group messages without labels

Imagine the bank had no intents yet, only messages. Could you discover the groups from the embeddings? Test it on six intents, whose true labels you hide from KMeans and only use to check the result.

1. Build a boolean mask of the training messages that belong to the six intents in `six`: `mask = train["category"].isin(six).to_numpy()`. Use it to keep those messages (`train[mask]`) and their vectors (`train_vectors[mask]`).
2. Cluster the vectors with `KMeans(n_clusters=6, n_init=10, random_state=0)`, and show a `pd.crosstab` of the clusters against the true intents.
3. Compute the adjusted Rand index (ARI) of the clustering.
4. Do the same with the TF-IDF rows of those messages (`train_tfidf[mask]`), and compare the two ARIs.

In [13]:
six = ["card_arrival", "lost_or_stolen_card", "exchange_rate", "top_up_failed", "cash_withdrawal_charge", "passcode_forgotten"]

# 1. Messages and vectors of the six intents
mask = train["category"].isin(six).to_numpy()
six_messages = train[mask].reset_index(drop=True)
six_vectors = train_vectors[mask]

# 2. KMeans and crosstab
kmeans = KMeans(n_clusters=6, n_init=10, random_state=0).fit(six_vectors)
six_messages["cluster"] = kmeans.labels_
display(pd.crosstab(six_messages["cluster"], six_messages["category"]))

# 3. ARI
print(f"ARI, embeddings: {adjusted_rand_score(six_messages['category'], kmeans.labels_):.2f}")

# 4. The same with TF-IDF
kmeans_tfidf = KMeans(n_clusters=6, n_init=10, random_state=0).fit(train_tfidf[mask])
display(pd.crosstab(kmeans_tfidf.labels_, six_messages["category"], rownames=["cluster"]))
print(f"ARI, TF-IDF:     {adjusted_rand_score(six_messages['category'], kmeans_tfidf.labels_):.2f}")

category,card_arrival,cash_withdrawal_charge,exchange_rate,lost_or_stolen_card,passcode_forgotten,top_up_failed
cluster,,,,,,
0,0,0,0,0,0,129
1,0,177,1,0,0,2
2,1,0,0,0,105,2
3,105,0,0,79,0,12
4,0,0,111,0,0,0
5,47,0,0,3,0,0


ARI, embeddings: 0.79


category,card_arrival,cash_withdrawal_charge,exchange_rate,lost_or_stolen_card,passcode_forgotten,top_up_failed
cluster,,,,,,
0,0,0,104,0,0,0
1,0,0,0,0,0,132
2,0,168,0,0,0,0
3,42,0,0,1,0,0
4,0,0,0,0,62,0
5,111,9,8,81,43,13


ARI, TF-IDF:     0.62


Investigating the two mixed embedding clusters (3 and 5) a bit more, a few messages per intent:

In [14]:
for cluster in [3, 5]:
    print(f"Cluster {cluster}")
    display(six_messages[six_messages["cluster"] == cluster].groupby("category").head(3)[["category", "text"]])

Cluster 3


,category,text
0,card_arrival,I am still waiting on my card?
1,card_arrival,What can I do if my card still hasn't arrived after 2 weeks?
2,card_arrival,I have been waiting over a week. Is the card still coming?
265,lost_or_stolen_card,Has there been any activity on my card today?
266,lost_or_stolen_card,I lost my wallet and all my cards were in it.
267,lost_or_stolen_card,I'm panicking! I lost my card! Help!
461,top_up_failed,My money I added isn't on my card.
468,top_up_failed,I really need to top-up my card today urgently but my card keeps getting declined!! Ca...
474,top_up_failed,My credit card did not go through. Can you see if it has insufficient funds?


Cluster 5


,category,text
3,card_arrival,Can I track my card while it is in the process of delivery?
8,card_arrival,Does the package with my card have tracking?
12,card_arrival,Can you track my card for me?
292,lost_or_stolen_card,Are you able to guide me to my card's location?
302,lost_or_stolen_card,I lost track of my card
321,lost_or_stolen_card,I misplaced my card and I dont know where the last place is where I used the card last...


**Checkpoint:** an ARI of about 0.8 for the embeddings and about 0.6 for TF-IDF.

**Question:** which intents does KMeans mix or split, and why might that happen? Would the clusters be useful to a bank that has no intents yet?

With the embeddings, 4 intents get their own clean cluster: `cash_withdrawal_charge` (177 of 177), `exchange_rate` (111 of 112), `passcode_forgotten` (105 of 105) and mostly `top_up_failed` (129 of 145). The problems are all around the card:

- **Mixed:** cluster 3 has 105 `card_arrival` and 79 `lost_or_stolen_card` messages together, plus 12 `top_up_failed` messages that mention a card (*I topped up my card but the app rejected it*). Seen from the customer, *my card hasn't arrived* and *I lost my card* are the same situation: I don't have my card and I need it. The messages use the same words and the same worry, only the reason differs.
- **Split:** `card_arrival` is split. Its other 47 messages form cluster 5, which is all about **tracking** the delivery (*Can I track my card?*, *Is there a way to track the delivery of my card?*). That's a real sub-topic, and KMeans found it on its own. Even the 3 `lost_or_stolen_card` messages in it fit (*I lost track of my card*). Since k is fixed at 6, it had to merge two other intents to make room for it.

TF-IDF is worse (ARI 0.62 vs 0.79): it finds 5 clean but incomplete clusters and dumps the rest into one big cluster with messages from every intent (111 `card_arrival`, 81 `lost_or_stolen_card`, 43 `passcode_forgotten` and a few of the others). `passcode_forgotten` is split in half (62 vs 43), probably by the words used (*passcode* vs *code* / *password*).

**Useful for a bank with no intents yet?** Yes, as a starting point, not as the final answer. The clean clusters can be named after reading a few messages each, and the "tracking" cluster shows that the clusters can find groups nobody thought of. But someone has to read samples of every cluster, try different k and decide what to split or merge. With 77 intents as close as `card_arrival` and `card_delivery_estimate` it would be a lot harder than with these six well-separated ones.

## Extra task: know when you don't know

A search tool should not pretend. When even the best match is not very similar, it is better to say *I'm not sure* and pass the message to a person.

1. For each test message, take the score of its best match (`similarity.max(axis=1)`).
2. For the thresholds 0.5, 0.6, 0.7, 0.8 and 0.9, compute (a) the share of test messages whose best score is at least the threshold (the messages the tool would answer) and (b) the top-1 accuracy on those messages only. Present the results as a table.
3. Which threshold would you recommend to the support team, and why?

Bonus: instead of taking the intent of the single best match, take the most common intent among the top 3, 5 and 10 matches (a majority vote, like the k-nearest neighbours classifier). Does it improve the accuracy? Then try a few `TfidfVectorizer` settings from Day 1 (`stop_words="english"`, `ngram_range=(1, 2)`): can any of them close the gap with the embeddings?

In [15]:
# Extra task
# 1. Score of the best match for every test message
best_score = similarity.max(axis=1)
correct = predicted == test["category"].to_numpy()

# 2. Share answered and accuracy for each threshold
rows = []
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    answered = best_score >= threshold
    rows.append({
        "threshold": threshold,
        "share answered": answered.mean(),
        "accuracy when answered": correct[answered].mean(),
        "accuracy of the ones passed to a person": correct[~answered].mean(),
    })
pd.DataFrame(rows).set_index("threshold").round(3)

,share answered,accuracy when answered,accuracy of the ones passed to a person
threshold,,,
0.5,0.999,0.921,0.333
0.6,0.997,0.922,0.500
0.7,0.979,0.929,0.545
0.8,0.894,0.944,0.719
0.9,0.604,0.964,0.854


The last column is not asked for, but it shows whether the threshold catches the right messages: the search's accuracy on the messages the tool would **not** answer.

**3. Recommendation:** the trade-off is clear. Up to 0.7 almost nothing changes: 98% of messages are answered and accuracy barely moves (0.921 → 0.929). From 0.8 it starts to work: the tool answers 89% of the messages with an accuracy of 0.944, and the 11% it passes to a person really are the hard ones (the search would only have been right on 72% of them). At 0.9 accuracy goes up to 0.964, but 40% of the messages go to a person, and the search would have been right on 85% of those anyway.

I would recommend **0.8**: about 1 in 10 messages goes to an agent, and wrong answers drop by almost a third (from 7.9% to 5.6% of answered messages). For intents where a wrong answer is expensive (stolen card, fraud), I would use a stricter threshold, or always pass them to a person, because of the *stolen* / *not stolen* problem from Challenge 1.

**Bonus 1: majority vote over the top k matches**

In [16]:
from collections import Counter

order = np.argsort(-similarity, axis=1)                 # training positions, most similar first
train_categories = train["category"].to_numpy()
for k in [1, 3, 5, 10]:
    votes = [Counter(row).most_common(1)[0][0] for row in train_categories[order[:, :k]]]
    print(f"top {k:>2}: accuracy {np.mean(np.array(votes) == test['category'].to_numpy()):.3f}")

top  1: accuracy 0.920
top  3: accuracy 0.924
top  5: accuracy 0.919
top 10: accuracy 0.915


Basically no improvement: top 3 gains 0.4 points (0.924), top 5 and top 10 are even slightly worse. With ~130 messages per intent the best match is usually already right. When it's wrong, the neighbours are mostly from the same confusing neighbour intent, so a vote doesn't help.

**Bonus 2: other TF-IDF settings**

In [17]:
settings = [
    {},
    {"stop_words": "english"},
    {"ngram_range": (1, 2)},
    {"stop_words": "english", "ngram_range": (1, 2)},
    {"ngram_range": (1, 2), "sublinear_tf": True},
]
for kwargs in settings:
    vectorizer = TfidfVectorizer(**kwargs)
    train_matrix = vectorizer.fit_transform(train["text"])
    best = np.asarray((vectorizer.transform(test["text"]) @ train_matrix.T).argmax(axis=1)).ravel()
    accuracy = (train_categories[best] == test["category"].to_numpy()).mean()
    print(f"{str(kwargs):<55} accuracy {accuracy:.3f}")

{}                                                      accuracy 0.791
{'stop_words': 'english'}                               accuracy 0.744


{'ngram_range': (1, 2)}                                 accuracy 0.735
{'stop_words': 'english', 'ngram_range': (1, 2)}        accuracy 0.728


{'ngram_range': (1, 2), 'sublinear_tf': True}           accuracy 0.737


None of the settings closes the gap: they all make TF-IDF **worse** than the default (0.73 to 0.74 instead of 0.79). Removing English stop words hurts, probably because in short messages like *my card has not arrived* the stop words (*not*, *my*, *has*, *where*) carry a big part of the meaning. Bigrams add many rare word pairs that rarely match between two messages. The gap to the embeddings (0.92) is not a tuning problem: TF-IDF only sees which words are shared, and customers describe the same problem with different words.

**Conclusions:**

- Semantic search with `all-MiniLM-L6-v2` finds the right intent for 92% of new messages, against 79% for TF-IDF. It wins because it matches meaning, not words (*billed me two times* → *charged twice*).
- Most of the remaining mistakes are neighbouring intents that a person would also mix up. The labels are part of the problem.
- Similarity is not meaning: *stolen* / *not stolen* score 0.96. For an automatic answer that's dangerous.
- A threshold of 0.8 on the best score lets the tool answer ~89% of messages at 94% accuracy and pass the unsure ones to a person.
- Without labels, KMeans on the embeddings recovers 4 of 6 intents cleanly (ARI 0.79) and even finds a sub-topic (card tracking). It merges intents that customers themselves don't separate.